In [12]:
!pip install -q langchain langchain-classic langchain-google-genai langchain-community rank-bm25 sentence-transformers

In [21]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pypdf sentence-transformers gradio

In [13]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"]=userdata.get("GOOGLE_API_KEY")


In [50]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2,
    max_retries=5,
)

In [52]:
from google.colab import files
uploaded=files.upload()
pdf_path=list(uploaded.keys())[0]
print("Uploaded:",pdf_path)

Saving lbdl.pdf to lbdl (2).pdf
Uploaded: lbdl (2).pdf


In [18]:
!pip install pypdf

In [53]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader(pdf_path)
pages=loader.load()
print("Total Pages: ",len(pages))

Total Pages:  189


In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings ready")

/tmp/ipykernel_917/2644305464.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings ready


In [54]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

parent_retriever = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embeddings),
    docstore=InMemoryStore(),
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 10},
)
parent_retriever.add_documents(pages)
print("Parent-child index ready")

Parent-child index ready


In [55]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

parent_chunks = parent_splitter.split_documents(pages)
bm25_retriever = BM25Retriever.from_documents(parent_chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=[0.4, 0.6],
)

In [56]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=4)

retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever,
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [62]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful teaching assistant. Answer using only the context below.\n\n"
     "Always write the answer in this exact order:\n"
     "1. **Definition:** start with a clear 1-2 sentence definition of the topic. "
     "Do not write anything before it.\n"
     "2. **Details:** then explain it step by step, with the formulas, how it works, "
     "and an example if the context has one.\n"
     "3. **Key points:** end with a short summary of 2-3 bullet points.\n\n"
     "Rules:\n"
     "- Mention page numbers like (Page 12).\n"
     "- Use only the context. If the answer is not in the context, "
     "say \"I couldn't find that in the PDF.\"\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

qa_chain = create_stuff_documents_chain(llm, answer_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

In [63]:
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

def to_messages(history):
    msgs = []
    for h in history:
        if isinstance(h, dict):
            cls = HumanMessage if h["role"] == "user" else AIMessage
            msgs.append(cls(content=h["content"]))
        else:
            msgs.append(HumanMessage(content=h[0]))
            msgs.append(AIMessage(content=h[1]))
    return msgs

def chat(message, history):
    result = rag_chain.invoke({"input": message, "chat_history": to_messages(history)})
    pages_used = sorted({d.metadata.get("page", 0) + 1 for d in result["context"]})
    return result["answer"] + f"\n\nSource pages: {pages_used}"

gr.ChatInterface(
    fn=chat,
    title="Advanced PDF RAG Chatbot",
    chatbot=gr.Chatbot(
        latex_delimiters=[
            {"left": "$$", "right": "$$", "display": True},
            {"left": "$", "right": "$", "display": False},
            {"left": "\\(", "right": "\\)", "display": False},
            {"left": "\\[", "right": "\\]", "display": True},
        ]
    ),
).launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://57790a87c1b55f3da1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://57790a87c1b55f3da1.gradio.live
